# Boltz2 Showcase

**BioPipelines example** — demonstrates all Boltz2 input types:
sequences, PDB structures, ligands, DNA, glycosylation, covalent linkages, and combinatorics (Bundle / Each).

[![Documentation](https://img.shields.io/badge/docs-readthedocs-blue)](https://biopipelines.readthedocs.io/en/latest/)
[![Preprint](https://img.shields.io/badge/preprint-bioRxiv-B31B1B)](https://www.biorxiv.org/content/10.64898/2026.03.11.711024v1)

In [ ]:
# Cell 1: Install BioPipelines and micromamba
!git clone https://github.com/locbp-uzh/biopipelines
%cd biopipelines
!pip install -e ".[all]"
!curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj -C /usr/local bin/micromamba
!micromamba create -f Environments/biopipelines.yaml -y

In [ ]:
# Cell 2: Install tools
from biopipelines.pipeline import *
from biopipelines.boltz2 import Boltz2

with Pipeline(project="Setup", job="InstallTools"):
    Boltz2.install()

In [ ]:
from biopipelines.pipeline import *
from biopipelines.boltz2 import Boltz2
from biopipelines.combinatorics import Bundle, Each

Pipeline(project="Examples", job="Boltz2", description="Boltz2 with various inputs")
# 1: Boltz2 with direct sequence
boltz_seq = Boltz2(
    proteins=Sequence("MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSH")
)
boltz_seq

In [ ]:
# 2: PDB fetch and Boltz2 with structure input
lysozyme = PDB("1AKI", ids="LYZ")
boltz_pdb = Boltz2(
    proteins=lysozyme
)
boltz_pdb

In [ ]:
# 3: Boltz2 with recycled MSAs
boltz_msa = Boltz2(
    proteins=lysozyme,
    msas=boltz_pdb
)
boltz_msa

In [ ]:
# 4: Boltz2 with ligand
boltz_ligand = Boltz2(
    proteins=Sequence("MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSH"),
    ligands=Ligand("ethanol")
)
boltz_ligand

In [ ]:
# 5: CompoundLibrary with multiple ligands
compounds = CompoundLibrary({
    'ethanol': 'CCO',
    'methanol': 'CO',
    'propanol': 'CCCO'
})
boltz_multi = Boltz2(
    proteins=lysozyme,
    ligands=compounds
)
boltz_multi

In [ ]:
# Setup for combinatorics examples
protein_a = Sequence("MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSH", ids="ProteinA")
protein_b = Sequence("MNIFEMLRIDEGLRLKIYKDTEGYYTIGIGHLLTKSPSLNAAKSELDKAIGRNTNGVITKDEAEKLFNQDVDAAVRGILRNAKLKPVYDSLDAVRRAALINMVFQMGETGVAGFTNSLRMLQQKRWDEAAVNLAKSRWYNQTPNRAKRVITTFRTGTWDAYKNL", ids="ProteinB")
ligand_library = Ligand(['aspirin', 'caffeine', 'ibuprofen'])

# 6: Each (Cartesian product) — 2 proteins × 3 ligands = 6 predictions
boltz_each = Boltz2(
    proteins=Each(protein_a, protein_b),
    ligands=ligand_library
)
boltz_each

In [ ]:
# 7: Bundle ligands — each protein gets all ligands together (2 predictions)
boltz_bundle_ligands = Boltz2(
    proteins=Each(protein_a, protein_b),
    ligands=Bundle(ligand_library)
)
boltz_bundle_ligands

In [ ]:
# 8: Bundle proteins — all proteins together with each ligand (3 predictions)
boltz_bundle_proteins = Boltz2(
    proteins=Bundle(protein_a, protein_b),
    ligands=ligand_library
)
boltz_bundle_proteins

In [ ]:
# 9: Bundle both — all proteins with all ligands in one prediction
boltz_bundle_all = Boltz2(
    proteins=Bundle(protein_a, protein_b),
    ligands=Bundle(ligand_library)
)
boltz_bundle_all

In [ ]:
# 10: Nested combinatorics — each library ligand bundled with ATP (3 predictions)
# Affinity is calculated for the library ligand (first in bundle)
atp = Ligand("ATP")
boltz_nested = Boltz2(
    proteins=protein_a,
    ligands=Bundle(Each(ligand_library), atp)
)
boltz_nested

In [ ]:
# 11: Nested combinatorics with multiple proteins — 2 proteins × 3 bundles = 6 predictions
boltz_nested_multi = Boltz2(
    proteins=Each(protein_a, protein_b),
    ligands=Bundle(Each(ligand_library), atp)
)
boltz_nested_multi

In [ ]:
# 12: Reversed bundle order — affinity calculated for ATP (first in bundle)
boltz_atp_affinity = Boltz2(
    proteins=protein_a,
    ligands=Bundle(atp, Each(ligand_library))
)
boltz_atp_affinity

In [ ]:
# 13: N-glycosylation on IgG1 Fc (Asn-297, position 73 in this fragment)
fc_fragment = Sequence(
    "TCPPCPAPELLGGPSVFLFPPKPKDTLMISRTPEVTCVVVDVSHEDPEVKFNWYVDGVEVHNAKTKPREEQYNSTYRVVSVLTVLHQDWLNGKEYKCKVSNKALPAPIEKTISKAKGQPREPQVYTLPPSRDELTKNQVSLTCLVKGFYPSDIAVEWESNGQPENNYKTTPPVLDSDGSFFLYSKLTVDKSRWQQGNVFSCSVMHEALHNHYTQKSLSLSPG",
    ids="IgG1_Fc"
)
boltz_glyco = Boltz2(
    proteins=fc_fragment,
    glycosylation={"A": [73]}
)
boltz_glyco

In [ ]:
# 14: Covalent linkage — benzylated SNAP-Tag (Cys-145)
# Step 1: run without covalent linkage to inspect atom numbering
SnapTag = Sequence("MDKDCEMKRTTLDSPLGKLELSGCEQGLHRIIFLGKGTSAADAVEVPAPAAVLGGPEPLMQATAWLNAYFHQPEAIEEFPVPALHHPVFQQESFTRQVLWKLLKVVKFGEVISYSHLAALAGNPAATAAVKTALSGNPVPILIPCHRVVQGDLDVGGYEGGLAVKEWLLAHEGHRLGKPGLG")
Bn = Ligand(smiles="CC1=CC=CC=C1")
boltz_inspection = Boltz2(
    proteins=SnapTag,
    ligands=Bn
)
boltz_inspection

In [ ]:
# 14 (cont.): Step 2 — run with covalent linkage (SG of Cys-145 to C15 of ligand)
boltz_covalent = Boltz2(
    proteins=SnapTag,
    ligands=Bn,
    msas=boltz_inspection,
    covalent_linkage={
        "chain": "A",
        "position": 145,
        "protein_atom": "SG",
        "ligand_atom": "C15"
    }
)
boltz_covalent

In [ ]:
# 15: Glycosylation + covalent linkage combined
boltz_glyco_covalent = Boltz2(
    proteins=SnapTag,
    ligands=Bn,
    msas=boltz_covalent,
    glycosylation={"A": [67]},
    covalent_linkage={
        "chain": "A",
        "position": 145,
        "protein_atom": "SG",
        "ligand_atom": "C15"
    }
)
boltz_glyco_covalent

In [ ]:
# 16: Contact constraints — Cys-145 (chain A) forced near C15 of ligand (chain B)
boltz_contact = Boltz2(
    proteins=SnapTag,
    ligands=Bn,
    msas=boltz_inspection,
    contacts=[
        {"token1": ["A", 145], "token2": ["B", "C15"], "max_distance": 8.0, "force": False}
    ]
)
boltz_contact

In [ ]:
# 17: Double-stranded DNA duplex
dna_strand = Sequence("GATTACAGATTACA", type="dna", ids="DNA_strand")
boltz_dna = Boltz2(
    dsDNA=dna_strand
)
boltz_dna

In [ ]:
# 18: Single-stranded DNA
boltz_ssdna = Boltz2(
    ssDNA=dna_strand
)
boltz_ssdna

In [ ]:
# 19: dsDNA + ligand — small molecule binding to DNA
dna_target = Sequence("AATTAATTAATTAATT", type="dna", ids="DNA_target")
daunorubicin = Ligand("daunorubicin")
boltz_dna_ligand = Boltz2(
    dsDNA=dna_target,
    ligands=daunorubicin
)
boltz_dna_ligand

In [ ]:
# 20: Protein + dsDNA + ligand — three-axis combinatorics (2 proteins × 2 ligands = 4 predictions)
boltz_three_axis = Boltz2(
    proteins=Each(protein_a, protein_b),
    dsDNA=dna_target,
    ligands=Each(Ligand("aspirin"), Ligand("caffeine"))
)
boltz_three_axis